# 1. Library Import

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 2. Data Load (샘플 데이터셋 가정)

In [ ]:
df = pd.read_csv('../data/marathon_results_2015.csv', index_col=0)

# 3. 데이터 전처리: 시간 데이터 변환 (HH:MM:SS -> Seconds)

In [ ]:
def time_to_seconds(time_str):
    if pd.isna(time_str) or str(time_str).strip() in ['-', '', 'nan']:
        return None
    parts = str(time_str).strip().split(':')
    if len(parts) == 3:
        return int(parts[0]) * 3600 + int(parts[1]) * 60 + int(parts[2])
    return None

split_cols = ['5K', '10K', '15K', '20K', 'Half', '25K', '30K', '35K', '40K', 'Official Time']
for col in split_cols:
    df[col + '_s'] = df[col].apply(time_to_seconds)

before = len(df)
df = df.dropna(subset=['Official Time_s', '5K_s', '10K_s', '30K_s', '40K_s'])
print(f"전처리 전: {before:,}행  →  전처리 후: {len(df):,}행  (제거: {before - len(df)}행)")

# 4. Feature Engineering: 피로도 지수(Fatigue Index) 생성
# 가설: 30K 이후 페이스 저하가 심할수록 완주 시간이 늦어질 것이다.
# 피로도 지수 = (30K~40K 평균 페이스) / (0K~10K 평균 페이스)
# 1보다 크면 후반부 페이스가 저하됨을 의미

In [ ]:
# Fatigue Index: 후반 10K 페이스(sec/km) / 초반 10K 페이스(sec/km)
# 1.0 = 페이스 동일, > 1.0 = 후반 느려짐 (히팅 더 월 지표)
df['Fatigue_Index'] = ((df['40K_s'] - df['30K_s']) / 10) / (df['10K_s'] / 10)

# Pacing Variance: 구간별 페이스 표준편차 (페이스 일관성 지표)
seg_paces = pd.DataFrame({
    '0-5K':   df['5K_s'] / 5,
    '5-10K':  (df['10K_s'] - df['5K_s']) / 5,
    '10-15K': (df['15K_s'] - df['10K_s']) / 5,
    '15-20K': (df['20K_s'] - df['15K_s']) / 5,
    '20-25K': (df['25K_s'] - df['20K_s']) / 5,
    '25-30K': (df['30K_s'] - df['25K_s']) / 5,
    '30-35K': (df['35K_s'] - df['30K_s']) / 5,
    '35-40K': (df['40K_s'] - df['35K_s']) / 5,
})
df['Pacing_Variance'] = seg_paces.std(axis=1)

df['Gender_bin'] = (df['M/F'] == 'M').astype(int)

print(f"Fatigue Index 평균: {df['Fatigue_Index'].mean():.3f}")
print(f"Fatigue Index > 1.1 (페이스 10% 저하): {(df['Fatigue_Index'] > 1.1).mean()*100:.1f}%")
print(f"Fatigue Index > 1.2 (페이스 20% 저하): {(df['Fatigue_Index'] > 1.2).mean()*100:.1f}%")

# 5. 시각화: 피로도 지수와 완주 시간의 상관관계

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(df['Fatigue_Index'], df['Official Time_s'] / 60, alpha=0.1, s=5, color='coral')
axes[0].set_title('Fatigue Index vs Official Finish Time')
axes[0].set_xlabel('Fatigue Index (후반/초반 페이스 비율)')
axes[0].set_ylabel('Finish Time (minutes)')
axes[0].axvline(x=1.0, color='gray', linestyle='--', alpha=0.7, label='FI=1.0 (페이스 유지)')
axes[0].legend()

seg_labels = ['0-5K','5-10K','10-15K','15-20K','20-25K','25-30K','30-35K','35-40K']
seg_means = [
    df['5K_s'].mean()/5,
    (df['10K_s']-df['5K_s']).mean()/5,
    (df['15K_s']-df['10K_s']).mean()/5,
    (df['20K_s']-df['15K_s']).mean()/5,
    (df['25K_s']-df['20K_s']).mean()/5,
    (df['30K_s']-df['25K_s']).mean()/5,
    (df['35K_s']-df['30K_s']).mean()/5,
    (df['40K_s']-df['35K_s']).mean()/5,
]
axes[1].bar(seg_labels, seg_means, color=['steelblue']*6 + ['tomato']*2)
axes[1].set_title('Average Pace per Segment (Hitting the Wall)')
axes[1].set_xlabel('Segment')
axes[1].set_ylabel('Pace (sec/km)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# 6. 상관계수 확인

In [ ]:
print("=== 주요 피처 간 상관계수 (Official Time 기준) ===")
corr_cols = ['Age', 'Gender_bin', 'Fatigue_Index', 'Pacing_Variance',
             '5K_s', '10K_s', '30K_s', 'Official Time_s']
corr = df[corr_cols].corr()['Official Time_s'].drop('Official Time_s').sort_values(ascending=False)
print(corr.round(3).to_string())

print("\n=== 최종 피처 목록 ===")
feature_cols = ['Age', 'Gender_bin', 'Fatigue_Index', 'Pacing_Variance',
                '5K_s', '10K_s', '15K_s', '20K_s', 'Half_s', '25K_s', '30K_s']
print(feature_cols)
print(f"\n총 학습 샘플 수: {df[feature_cols + ['Official Time_s']].dropna().__len__():,}")